In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler #for preprocession
from sklearn.model_selection import train_test_split #split the train and test
from sklearn.linear_model import LogisticRegression #for run the logistic regression
from sklearn.model_selection import GridSearchCV # for hypertune at once
from sklearn.metrics import confusion_matrix # to get the value in matrix form
from sklearn.metrics import classification_report # to get the overall performance
from sklearn.metrics import f1_score # to get the f1 score value
from sklearn.metrics import roc_auc_score # to get the score of receiver operating curve and areaunder curve

In [3]:
dataset = pd.read_csv('Social_Network_Ads.csv')#dataset collection

In [4]:
#dataset preprocessing by converting the nominal into ordinal 
dataset = pd.get_dummies(dataset, drop_first = True, dtype = int)
dataset.drop ('User ID', axis = 1) #we cant put any test input for user id column hence its removed

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [6]:
#separate input and output
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [7]:
input = dataset[['Age', 'EstimatedSalary','Gender_Male']]
output = dataset[['Purchased']]

In [11]:
#separate train and test data
x_train, x_test, y_train, y_test = train_test_split(input, output, test_size = 1/3, random_state = 0)

In [9]:
#dataset preprocessing by standardscaler
#standardize the  data by converting the exist value into certain range (0 to 1)
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)


In [10]:
dataset[['Purchased']].value_counts() #dataset is highly imbalanced

Purchased
0            257
1            143
Name: count, dtype: int64

In [14]:
#hypertune by Gridsearch CV
parameter = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': [1.0],
    'l1_ratio': [0.0],
    'dual': [False],
    'tol': [0.0001],
    'fit_intercept': [True],
    'intercept_scaling': [1],
    'class_weight': [None],
    'random_state': [None],
    'solver': ['lbfgs'],
    'max_iter': [100],
    'verbose': [0],
    'warm_start': [False],
    'n_jobs': [None]
}
grid = GridSearchCV(LogisticRegression(), parameter, refit = True, verbose = 3, n_jobs = -1, scoring = 'f1_weighted')
grid.fit(x_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
10 fits failed out of a total of 15.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\HP\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\HP\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1218, in fit
    solver = _check_solver(

,estimator,LogisticRegression()
,param_grid,"{'C': [1.0], 'class_weight': [None], 'dual': [False], 'fit_intercept': [True], ...}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [16]:
result = grid.cv_results_
grid_pred = grid.predict(x_test)

#confusion matrix
cm = confusion_matrix(y_test, grid_pred)

#classification report generate
clf = classification_report (y_test, grid_pred)

print(cm)
print(clf)


[[80  5]
 [14 35]]
              precision    recall  f1-score   support

           0       0.85      0.94      0.89        85
           1       0.88      0.71      0.79        49

    accuracy                           0.86       134
   macro avg       0.86      0.83      0.84       134
weighted avg       0.86      0.86      0.85       134



In [19]:
#generate f1 score value
f1_macro=f1_score(y_test,grid_pred,average='weighted')
print ("The f1_macro value for best parameter{}.".format(grid.best_params_), f1_macro)

The f1_macro value for best parameter{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 100, 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}. 0.8546043244326981


In [22]:
#generate roc_auc_score value
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

0.9457382953181273